# Komputasi Saintifik — Studi Kasus EGIER
**Cornelius Nathanael Bedu — 2902594033 — Cybersecurity**

Catatan pendekatan: data dipertahankan pada resolusi dwimingguan aslinya (M1–M144, 24 periode/tahun, M1 = paruh pertama Januari 2018). Pemilihan metode di tiap soal dijelaskan langsung di selnya. Unggah `aol_data.xlsx` ke sesi Colab lebih dulu.

In [ ]:
import numpy as np, pandas as pd, math
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import simpson, trapezoid

df = pd.read_excel('aol_data.xlsx', sheet_name='in')
y  = df.iloc[0].values.astype(float)
t  = np.arange(1, len(y) + 1, dtype=float)
n  = len(y)
print('Jumlah periode:', n, '| rentang nilai:', y.min(), '-', y.max(), '| total:', y.sum())

## Soal 1 — Model tren (regresi eksponensial dua parameter)
Sebaran data menanjak makin curam, jadi pendekatan linear tidak dipakai. Untuk menguji apakah pola itu eksponensial, saya pakai **diagnostik log-linear**: kalau ln(y) hampir lurus terhadap M, berarti y mengikuti bentuk a·e^(bt). Model akhir hanya memakai dua parameter (tanpa konstanta offset) supaya sehemat mungkin.

In [ ]:
# Diagnostik: apakah ln(y) linear terhadap t?
sl, ic = np.polyfit(t, np.log(y), 1)
lin_pred = np.polyval((sl, ic), t)
R2_log = 1 - np.sum((np.log(y) - lin_pred)**2) / np.sum((np.log(y) - np.log(y).mean())**2)
print(f'R^2 garis ln(y) vs t = {R2_log:.5f}  -> sangat lurus, mendukung bentuk eksponensial')

def model(x, a, b):
    return a * np.exp(b * x)

(a, b), _ = curve_fit(model, t, y, p0=[np.exp(ic), sl], maxfev=10000)
pred = model(t, a, b)
R2 = 1 - np.sum((y - pred)**2) / np.sum((y - y.mean())**2)
rmse = np.sqrt(np.mean((y - pred)**2))
print(f'Model: y(t) = {a:.3f} * exp({b:.6f} * t)')
print(f'R^2 = {R2:.5f} | RMSE = {rmse:.3f} | laju tumbuh per periode = {(math.exp(b)-1)*100:.3f}%')

In [ ]:
fig, (p1, p2) = plt.subplots(1, 2, figsize=(10, 3.6))
p1.scatter(t, y, s=12, alpha=0.5); p1.plot(t, pred, 'r-', lw=2)
p1.set_title('Data dan model'); p1.set_xlabel('M'); p1.set_ylabel('Produksi'); p1.grid(alpha=0.3)
p2.scatter(t, np.log(y), s=12, alpha=0.5); p2.plot(t, lin_pred, 'k-', lw=1.6)
p2.set_title('Diagnostik log-linear'); p2.set_xlabel('M'); p2.set_ylabel('ln(Produksi)'); p2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Soal 2 — Bentuk numerik lewat reduksi rentang + Taylor
Deret Maclaurin untuk $e^z$ melambat kalau $z$ besar; di data ini $z=bt$ mencapai ~2,17. Cara yang dipakai pustaka nyata (dan yang saya tiru) adalah **reduksi rentang**: tulis $z = k\ln 2 + r$ dengan $k$ bulat dan $|r| \le \tfrac{1}{2}\ln 2 \approx 0,347$, lalu

$$e^z = 2^{k}\,e^{r}, \qquad e^{r} = \sum_{i=0}^{N}\frac{r^{i}}{i!}.$$

Karena $|r|$ kecil, deret untuk $e^r$ menyatu sangat cepat sehingga suku yang dibutuhkan jauh lebih sedikit.

In [ ]:
ln2 = math.log(2)
def exp_reduksi(z, N):
    k = round(z / ln2)
    r = z - k * ln2                      # |r| <= ln2/2
    s = sum(r**i / math.factorial(i) for i in range(N + 1))
    return (2.0**k) * s

print(f'Batas |r| = ln2/2 = {ln2/2:.4f}')
for N in range(1, 7):
    galat = max(abs(exp_reduksi(b*tt, N) - math.exp(b*tt)) / math.exp(b*tt) for tt in t)
    print(f'N={N}: galat relatif maksimum di seluruh data = {galat:.3e}')
print('-> N=3 sudah memenuhi presisi tiga digit (1e-3); N=5 mencapai ~1e-6.')

In [ ]:
# Visual Soal 2: galat menurun terhadap jumlah suku
Ns = list(range(1, 8))
galat = [max(abs(exp_reduksi(b*tt, N) - math.exp(b*tt)) / math.exp(b*tt) for tt in t) for N in Ns]
plt.figure(figsize=(7, 3.4))
plt.semilogy(Ns, galat, 'o-', lw=1.9)
plt.axhline(1e-3, ls='--', color='r', label='ambang presisi 3 digit (1e-3)')
plt.xlabel('Jumlah suku Taylor pada argumen tereduksi (N)')
plt.ylabel('Galat relatif maksimum')
plt.title('Konvergensi reduksi-rentang + Taylor')
plt.legend(); plt.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()

## Soal 3 — Kapan kapasitas 25.000 tercapai (bisection + cek analitik)
Saya pakai **metode bagi dua (bisection)** sebagai metode utama: tidak butuh turunan, dijamin menyatu selama akar terkurung dalam selang, dan cocok karena $f(t)=a\,e^{bt}-25000$ jelas berganti tanda. Karena modelnya eksponensial murni, akarnya juga punya bentuk tertutup $t=\ln(25000/a)/b$, yang saya pakai untuk memverifikasi hasil numerik. Membangun gudang butuh 13 bulan = 26 periode.

In [ ]:
C = 25000.0
f = lambda x: a * math.exp(b * x) - C

lo, hi = 100.0, 250.0
assert f(lo) * f(hi) < 0
it = 0
while hi - lo > 1e-9:
    mid = (lo + hi) / 2
    if f(lo) * f(mid) <= 0:
        hi = mid
    else:
        lo = mid
    it += 1
t_root = (lo + hi) / 2
t_exact = math.log(C / a) / b
print(f'Bisection: t* = {t_root:.4f} periode dalam {it} iterasi')
print(f'Cek analitik t = ln(C/a)/b = {t_exact:.6f}  (selisih {abs(t_root-t_exact):.2e})')

def bulan(tp):
    mi = 2018*12 + (tp - 1) * 0.5
    yr = int(mi // 12); mo = int(mi - yr*12)
    nama = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
    return f'{nama[mo]} {yr}'
print(f'Kapasitas penuh : {bulan(t_root)} (M{t_root:.2f})')
print(f'Mulai bangun    : {bulan(t_root - 26)} (M{t_root - 26:.2f})')

In [ ]:
# Visual Soal 3: kurva model menembus kapasitas
xx = np.linspace(1, 205, 500)
plt.figure(figsize=(7, 3.4))
plt.plot(xx, model(xx, a, b), lw=2, label='Model tren')
plt.axhline(C, ls='--', color='r', label='Kapasitas 25.000')
plt.axvspan(t_root - 26, t_root, alpha=0.12, color='purple')
plt.scatter([t_root], [C], color='r', zorder=5)
plt.scatter([t_root - 26], [model(t_root - 26, a, b)], color='purple', zorder=5)
plt.annotate(f'penuh M{t_root:.0f}\n~{bulan(t_root)}', (t_root, C),
             xytext=(8, -28), textcoords='offset points', fontsize=8, color='r')
plt.annotate(f'mulai bangun M{t_root-26:.0f}\n~{bulan(t_root-26)}', (t_root - 26, model(t_root - 26, a, b)),
             xytext=(-95, 14), textcoords='offset points', fontsize=8, color='purple')
plt.xlabel('Periode M'); plt.ylabel('Produksi (unit)')
plt.title('Akar bisection: ambang kapasitas & jendela 13 bulan')
plt.legend(loc='upper left'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Soal 4a — Laju perubahan pada resolusi dwimingguan asli
Berbeda dari meratakan dulu ke bulanan, saya menghitung turunan langsung pada langkah dwimingguan dengan selisih terpusat supaya variasi antarperiode tidak hilang. 2020 = M49–M72, 2021 = M73–M96.

In [ ]:
laju = np.gradient(y, t)   # selisih terpusat orde-2
def bulan(tp):
    mi = 2018*12 + (tp - 1) * 0.5
    yr = int(mi // 12); mo = int(mi - yr*12)
    nama = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
    return f'{nama[mo]} {yr}'
for label, (s, e) in {'2020': (49, 72), '2021': (73, 96)}.items():
    idx = np.arange(s, e + 1); d = laju[s - 1:e]
    up, dn = idx[d.argmax()], idx[d.argmin()]
    print(f'{label}: naik tercuram di M{up} (+{d.max():.1f}/periode, ~{bulan(up)}) | '
          f'turun tercuram di M{dn} ({d.min():.1f}/periode, ~{bulan(dn)})')

In [ ]:
# Visual Soal 4a: laju perubahan dwimingguan 2020 & 2021
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for k, (label, (s, e)) in enumerate({'2020': (49, 72), '2021': (73, 96)}.items()):
    idx = np.arange(s, e + 1); d = laju[s - 1:e]
    ax[k].bar(idx, d, alpha=0.65, width=0.8)
    ax[k].scatter([idx[d.argmax()]], [d.max()], color='orange', zorder=5, s=30)
    ax[k].scatter([idx[d.argmin()]], [d.min()], color='black', zorder=5, s=30)
    ax[k].axhline(0, color='k', lw=0.6); ax[k].set_title(label); ax[k].set_xlabel('Periode M'); ax[k].grid(alpha=0.2, axis='y')
ax[0].set_ylabel('Laju (unit/periode)')
fig.suptitle('Laju perubahan resolusi dwimingguan (oranye = naik tercuram, hitam = turun tercuram)', fontsize=9)
plt.tight_layout(); plt.show()

## Soal 4b — Total produksi (jepit Riemann kiri/kanan, lalu trapezoidal & Simpson)
Karena data naik, jumlah Riemann kiri pasti meremehkan dan Riemann kanan melebihkan luas; aturan trapezoidal adalah rata-rata keduanya. Saya pakai ini untuk menjepit nilai sebenarnya sekaligus menjelaskan dari mana selisih terhadap penjumlahan langsung berasal.

In [ ]:
aktual = y.sum()
Lk = y[:-1].sum()        # Riemann kiri
Rk = y[1:].sum()         # Riemann kanan
trap = trapezoid(y, dx=1.0)
simp = simpson(y, dx=1.0)
for nama, v in [('Penjumlahan langsung', aktual), ('Riemann kiri', Lk),
                ('Riemann kanan', Rk), ('Trapezoidal', trap), ('Simpson 1/3', simp)]:
    sel = '' if nama == 'Penjumlahan langsung' else f'  ({(v-aktual)/aktual*100:+.3f}%)'
    print(f'{nama:22s}: {v:>12,.1f}{sel}')
print(f'\nTrapezoidal = (kiri + kanan)/2 = {(Lk+Rk)/2:,.1f}, '
      f'selisihnya terhadap jumlah = setengah titik ujung = {(y[0]+y[-1])/2:.1f}')

In [ ]:
# Visual Soal 4b: perbandingan taksiran total terhadap penjumlahan langsung
nama = ['Penjumlahan\nlangsung', 'Riemann\nkiri', 'Riemann\nkanan', 'Trapezoidal', 'Simpson 1/3']
nilai = [aktual, Lk, Rk, trap, simp]
plt.figure(figsize=(7, 3.4))
bars = plt.bar(nama, nilai, alpha=0.75)
bars[0].set_color('#0d9488')
plt.axhline(aktual, ls='--', color='gray', lw=1)
for x, v in zip(nama, nilai):
    sel = '' if v == aktual else f'{(v-aktual)/aktual*100:+.2f}%'
    plt.text(x, v, sel, ha='center', va='bottom', fontsize=8)
plt.ylim(min(nilai)*0.985, max(nilai)*1.01)
plt.ylabel('Total produksi (unit)'); plt.title('Total enam tahun: jepit Riemann vs aturan kuadratur')
plt.tight_layout(); plt.show()